**Part 5 · Projects** · step 23 of 23  
[← 22 Project: dynamic tool agent (part 1)](01_db_schema_design.ipynb) · [🏠 Roadmap](../../README.md) · 🎉 end of course

# 22 · Project — Dynamic Tool Agent, Part 2: Build agents from the database

1. Tables `agents`, `tools`, `agent_tools` live in `agent.db`.
2. A tool row stores Python source code with an `execute(...)` function.
3. `load_agent(agent_id)` reads the rows, turns each code string into a LangChain tool
   with `StructuredTool.from_function`, and calls `create_agent`.

⚠️ **Security:** `exec()` runs whatever code is in the database. Only acceptable for trusted,
local learning data — a real system needs sandboxing (separate process/container, timeouts).

In [ ]:
from dotenv import load_dotenv

# Reads API keys (OPENAI_API_KEY, GROQ_API_KEY, GOOGLE_API_KEY, TAVILY_API_KEY)
# from the .env file at the AgenticAI root. load_dotenv() walks up parent
# folders until it finds one, so every notebook can share a single .env.
load_dotenv()

## Database helpers

In [ ]:
import sqlite3

DB = "agent.db"


def get_db() -> sqlite3.Connection:
    """Open agent.db; rows behave like dicts (row["name"])."""
    conn = sqlite3.connect(DB)
    conn.row_factory = sqlite3.Row
    return conn


def create_tables() -> None:
    """Create the three tables if they don't exist yet."""
    with get_db() as conn:
        conn.executescript(
            """
            CREATE TABLE IF NOT EXISTS agents (
                id TEXT PRIMARY KEY,
                name TEXT,
                model TEXT,
                system_prompt TEXT
            );
            CREATE TABLE IF NOT EXISTS tools (
                id TEXT PRIMARY KEY,
                name TEXT,
                description TEXT,
                type TEXT DEFAULT 'script',
                code TEXT,
                entry_function TEXT DEFAULT 'execute'
            );
            CREATE TABLE IF NOT EXISTS agent_tools (
                agent_id TEXT,
                tool_id TEXT,
                PRIMARY KEY(agent_id, tool_id)
            );
            """
        )


def create_agent_db(agent_id: str, name: str, model: str, system_prompt: str) -> None:
    """Insert an agent definition (ignored if the id already exists)."""
    with get_db() as conn:
        conn.execute("INSERT OR IGNORE INTO agents VALUES (?, ?, ?, ?)", (agent_id, name, model, system_prompt))


def create_tool(tool_id: str, name: str, description: str, code: str) -> None:
    """Insert a tool whose ``code`` defines an ``execute`` function."""
    with get_db() as conn:
        conn.execute(
            "INSERT OR IGNORE INTO tools (id, name, description, code) VALUES (?, ?, ?, ?)",
            (tool_id, name, description, code),
        )


def bind_tool(agent_id: str, tool_id: str) -> None:
    """Give an agent access to a tool."""
    with get_db() as conn:
        conn.execute("INSERT OR IGNORE INTO agent_tools VALUES (?, ?)", (agent_id, tool_id))

## Seed data (safe to re-run — duplicates are ignored)

In [ ]:
create_tables()

create_tool("weather", "get_weather", "Get the weather of a city", '''
def execute(city: str) -> str:
    "Get the weather of a city."
    return f"Weather in {city} is Sunny"
''')

create_tool("calculator", "add_numbers", "Add two numbers", '''
def execute(a: int, b: int) -> int:
    "Add two numbers."
    return a + b
''')

create_agent_db("assistant", "This is an AI Assistant", "gpt-4o", "You are a helpful assistant.")
bind_tool("assistant", "weather")
bind_tool("assistant", "calculator")

## Turn DB rows into a LangChain agent

Every stored function is called `execute`. If we passed the raw functions to the agent, **both tools
would be named `execute`** and the model couldn't tell them apart. `StructuredTool.from_function`
lets us give each one the `name` and `description` stored in the database.

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import StructuredTool


def build_tool(tool_row: sqlite3.Row) -> StructuredTool:
    """Compile the tool's source code and wrap its entry function as a LangChain tool."""
    namespace: dict = {}
    exec(tool_row["code"], namespace)  # defines execute() inside `namespace`
    func = namespace[tool_row["entry_function"]]
    return StructuredTool.from_function(func, name=tool_row["name"], description=tool_row["description"])


def load_agent(agent_id: str):
    """Read an agent and its tools from the DB and build a runnable agent."""
    with get_db() as conn:
        agent_row = conn.execute("SELECT * FROM agents WHERE id = ?", (agent_id,)).fetchone()
        tool_rows = conn.execute(
            """
            SELECT t.* FROM tools t
            JOIN agent_tools at ON t.id = at.tool_id
            WHERE at.agent_id = ?
            """,
            (agent_id,),
        ).fetchall()

    tools = [build_tool(row) for row in tool_rows]
    print("Loaded tools:", [t.name for t in tools])

    return create_agent(model=agent_row["model"], system_prompt=agent_row["system_prompt"], tools=tools)

In [ ]:
agent = load_agent("assistant")

result = agent.invoke({"messages": [{"role": "user", "content": "What is the weather in Delhi? Also what is 12 + 30?"}]})
result["messages"][-1].content

---
✅ **Done with 22 Project: dynamic tool agent (part 2).** Next: 🎉 end of course  
[🏠 Roadmap](../../README.md)